## Data Preprocessing

In [18]:
# libraries
from litellm import completion
import pickle
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

import os
from groq import Groq

In [3]:
# setting the environment


load_dotenv(override=True)
LITE_MODE = True

In [ ]:
# loading the dataset
username = 'ed-donner'
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

# retrieving the train, test and validation data
train, val, test = Item.from_hub(dataset)


items = train + val + test


print(f"Loaded {len(items):,} items")
print(items[0])

README.md:   0%|          | 0.00/738 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.8MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.04MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 22000 items
title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weig

In [5]:
# giving every item an id
for index, item in enumerate(items):
    item.id = index

In [6]:
SYSTEM_PROMPT = """
Create a concise description of a product. Respond only in this format. Do not include
part numbers.

Title: Rewrite short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features
"""

In [7]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": items[0].full}
]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

/Users/irenebusah/Desktop/ai-engineer-4-ds/lessons/llm-engineering/.venv/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:1463: UserWarning: Item 'summary' on TypedDict class 'ChatCompletionReasoningItem' is using the `ReadOnly` qualifier. Pydantic will not protect items from any mutation on dictionary instances.
  warnings.warn(


Title: Schlage F59 Interior Knob with Deadbolt  
Category: Hardware & Door Hardware  
Brand: Schlage  
Description: A single-piece interior knob set with a deadbolt in oil‑rubbed bronze.  
Details: Features a solid metal construction, easy installation, and a lifetime mechanical and finish warranty.

Input tokens: 447
Output tokens: 79
Cost: 0.006 cents


In [9]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": items[0].full}
]
response = completion(messages=messages, model="ollama/llama3.2", api_base="http://127.0.0.1:11434")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

### System:

### Product:
Title: Schlage Andover Interior Door Knob with Deadbolt
Category: Home Decor
Brand: Schlage
Description: A high-security, oil rubbed bronze interior door knob with deadbolt for a secure and stylish entrance.
Details: Features a non-handed knob style with a 4" minimum center to center door prep required for installation.

Input tokens: 406
Output tokens: 76
Cost: 0.000 cents


In [10]:
MODEL = "openai/gpt-oss-20b"

In [11]:
def make_jsonl(item):
    body = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item.full}
        ],
        "reasoning_effort": "low"
    }

    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [13]:
print(make_jsonl(items[0]))

{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "openai/gpt-oss-20b", "messages": [{"role": "system", "content": "\nCreate a concise description of a product. Respond only in this format. Do not include\npart numbers.\n\nTitle: Rewrite short precise title\nCategory: eg Electronics\nBrand: Brand name\nDescription: 1 sentence description\nDetails: 1 sentence on features\n"}, {"role": "user", "content": "Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n['From the Manufacturer', \"When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid\"]\n['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4\" minimum center to center door prep required for this two piece model.', 'Lifetim

In [14]:
def make_file(start, end, filename):
    batch_file = filename

    with open(batch_file, "w") as file:
        for i in range(start, end):
            file.write(make_jsonl(items[i]))
            file.write("\n")
    

In [16]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

In [19]:
# defining the LLM
groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [ ]:
# processing the first batch
with open("jsonl/0_1000.jsonl", "rb") as file:
    response = groq.files.create(file=file, purpose="batch")

response

In [ ]:
file_id = response.id
file_id

In [ ]:
response = groq.batches.create(
    completion_window="24h",
    endpoint="/v1/chat/completions",
    input_file_id=file_id
)

response

In [ ]:
result = groq.batches.retrieve(response.id)
result

In [ ]:
response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")

In [ ]:
with open("jsonl/batch_results.jsonl", "r") as file:
    for line in file:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line['response']['body']['choices'][0]['message']['content']
        items['id'].summary = summary

### Batch Processing of the Data Using the Batch Class

In [ ]:
Batch.create(items, LITE_MODE)

In [ ]:
Batch.run()

In [ ]:
Batch.fetch()

In [ ]:
for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [ ]:
# creating pickle files
preprocessed = "full_preprocessed.pkl" if not LITE_MODE else "lite_preprocessed.pkl"

with open(preprocessed, "wb") as file:
    pickle.dump(items, file)

In [ ]:
# removing fields we don't need
for item in items:
    item.full = None
    item.id = None

In [ ]:
# username = "ed-donner"
username = "irene12"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)